# 24 – Supervisor Agent (Legacy Orchestrator)

The `SupervisorAgent` is the legacy orchestrator kept for backward compatibility.
In production, the LangGraph pipeline supersedes it, but it remains useful for:
- Simple single-agent dispatch without the full graph overhead
- Direct agent testing without building a graph
- Legacy client compatibility

**Note:** For new code, prefer `graph.invoke()` (see notebook 08).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

In [ ]:
from agents.supervisor_agent import SupervisorAgent, SupervisorResponse
from core.base_agent import AgentRequest

supervisor = SupervisorAgent()

## 1. execute() — redirect notice

In [ ]:
req = AgentRequest(query='What is the retention GRR?')
result = supervisor.execute(req)
print('Success :', result.success)
print('Message :', result.message)

## 2. run() — legacy dispatch interface

In [ ]:
queries = [
    'list all rules',
    'list rules for retention',
    'create rule: GRR must exceed 85%',
    'What is the governance policy for bookings?',
]

for q in queries:
    resp = supervisor.run(q)
    print(f"Query  : '{q}'")
    print(f"Success: {resp.success}")
    print(f"Agents : {resp.agents_used}")
    print(f"Summary: {resp.summary[:80]}")
    print()

## 3. SupervisorResponse dataclass

In [ ]:
# Direct construction
resp = SupervisorResponse(
    success=True,
    summary='DQ score for bookings is 94.5%, above the 90% threshold.',
    agents_used=['metadata', 'information'],
    data={'dq_score': 94.5, 'product': 'bookings'},
    confidence=0.92,
)

print('success    :', resp.success)
print('agents_used:', resp.agents_used)
print('confidence :', resp.confidence)
print('data       :', resp.data)

## 4. Routing keywords reference

In [ ]:
print('Rule-routing keywords     :', SupervisorAgent._RULE_KEYWORDS)
print('Rule-only keywords        :', SupervisorAgent._RULE_ONLY_KEYWORDS)

# Show which queries route to rule agent
test = [
    'list all rules',
    'create rule: bookings null check',
    'define governance policy',
    'show DQ score',
]

print('\nRouting decisions:')
for q in test:
    ql = q.lower()
    routes_rule = any(kw in ql for kw in SupervisorAgent._RULE_KEYWORDS)
    print(f"  '{q}' => {'rule_agent' if routes_rule else 'default_agents'}")